# Sims GUI + modules

Dit notebook houdt de GUI overzichtelijk en gebruikt de functies uit de losse `.py`-bestanden.

**Flow:** GUI → spelerinstructie → safety → LangChain/RAG → ethics → actie uitvoeren.

In [1]:
# Alleen nodig als je bestanden nog namen hebben zoals "tijdreis_rag(1).py".
# In je eigen branch kun je deze cel overslaan als de bestanden al netjes heten.
from pathlib import Path
import shutil

ALIASES = {
    "langchain_logic_100(1).py": "langchain_logic_100.py",
    "tijdreis_rag(1).py": "tijdreis_rag.py",
    "safety_guard(1).py": "safety_guard.py",
    "ethics_engine(1).py": "ethics_engine.py",
    "prehistorie(1).txt": "werelden/prehistorie.txt",
    "toekomst(1).txt": "werelden/toekomst.txt",
}

for source, target in ALIASES.items():
    source_path = Path(source)
    target_path = Path(target)
    if source_path.exists() and not target_path.exists():
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(source_path, target_path)

Path("karakters").mkdir(exist_ok=True)
for naam in ["Lars", "Emma", "Fatima", "Daan", "Sofia"]:
    path = Path("karakters") / f"{naam.lower()}.txt"
    if not path.exists():
        path.write_text(f"{naam} is een vriendelijke Sim die graag samenwerkt en kindvriendelijke keuzes maakt.", encoding="utf-8")


In [2]:
import random
import tkinter as tk
from tkinter import simpledialog

from langchain_logic_100 import kies_actie_voor_sim_dict, registreer_resultaat
from safety_guard import filter_speler_instructie, filter_llm_tekst
from ethics_engine import beschrijf_ethische_keuze, score_actie

try:
    from news_logic import breng_nieuws_naar_dorp
except Exception:
    breng_nieuws_naar_dorp = None


/Users/jip/Documents/Semester_6/GenAI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
GRID_SIZE = 15
CELL_SIZE = 40
DEFAULT_SPEED = 1000
TIJDPERK = "prehistorie"   # kies: "prehistorie" of "toekomst"
GEBRUIK_RAG = True          # zet tijdelijk op False als Ollama/RAG nog niet werkt

SIM_ICON = "🙂"
OBJECTEN = {
    "🌳": "boom",
    "🛌": "bed",
    "🍏": "appel",
    "🎶": "radio",
    "🔥": "kampvuur",
    "💤": "slaapplek",
}


## GUI basis

Dit is alleen de GUI-laag. De LLM/RAG/ethiek-logica staat in de losse modules en wordt pas in `MijnSimsWereld` aangeroepen.

In [7]:
class SimsWereld:
    def __init__(self, root):
        self.root = root
        self.root.title("LLM Sims")
        self.canvas = tk.Canvas(root, width=GRID_SIZE * CELL_SIZE, height=GRID_SIZE * CELL_SIZE)
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        self.canvas.pack()

        self.speed = DEFAULT_SPEED
        self.running = False
        self.step_mode = False
        self.stopping = False
        self.grid = [[{} for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]

        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        self.initialiseer_wereld()
        self.draw_grid()
        self.create_buttons()
        self.root.after(self.speed, self.update_world)

    def on_close(self):
        self.stopping = True
        self.root.destroy()

    def draw_grid(self):
        self.canvas.delete("all")
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                x1, y1 = x * CELL_SIZE, y * CELL_SIZE
                x2, y2 = x1 + CELL_SIZE, y1 + CELL_SIZE
                self.canvas.create_rectangle(x1, y1, x2, y2, fill="white", outline="gray")
                if self.grid[y][x]:
                    self.canvas.create_text(
                        x1 + CELL_SIZE // 2,
                        y1 + CELL_SIZE // 2,
                        text=self.grid[y][x]["icon"],
                        font=("Arial", 16),
                    )

    def create_buttons(self):
        frame = tk.Frame(self.root)
        frame.pack()
        tk.Button(frame, text="▶️ Play/Pauze", command=self.toggle_play).grid(row=0, column=0)
        tk.Button(frame, text="⏭️ Stap", command=self.step).grid(row=0, column=1)
        tk.Button(frame, text="📢 Instructie", command=self.instructie_voor_alle_sims).grid(row=0, column=2)
        tk.Button(frame, text="📰 Nieuws", command=self.nieuws_api_demo).grid(row=0, column=3)

    def toggle_play(self):
        self.step_mode = False
        self.running = not self.running

    def step(self):
        self.step_mode = True
        self.running = True

    def update_world(self):
        if self.running:
            self.doe_alles()
            self.draw_grid()
            if self.step_mode:
                self.running = False
        if not self.stopping:
            self.root.after(self.speed, self.update_world)

    def instructie_voor_alle_sims(self):
        instructie = simpledialog.askstring("Instructie", "Geef alle Sims een opdracht:")
        if instructie:
            for sim, _, _ in self.vind_sims():
                self.instrueer(sim, instructie)

    def on_canvas_click(self, event):
        x = event.x // CELL_SIZE
        y = event.y // CELL_SIZE
        if 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE and self.grid[y][x].get("type") == "speler":
            instructie = simpledialog.askstring("Instructie", "Geef deze Sim een opdracht:")
            if instructie:
                self.instrueer(self.grid[y][x], instructie)

    def instrueer(self, sim, instructie):
        sim["instructies"].append(filter_speler_instructie(instructie))

    def nieuws_api_demo(self):
        print("API-demo is optioneel. Koppel hier je eigen nieuws/API-input aan news_logic.py.")

    def initialiseer_wereld(self):
        raise NotImplementedError

    def doe_alles(self):
        raise NotImplementedError


## Spel + koppeling met modules

Hier zit de eigenlijke integratie: de GUI vraagt één beslissing aan `kies_actie_voor_sim_dict(...)` en voert die actie uit.

In [8]:
def ethisch_advies(sim, actie, objecten_nabij, tijdperk):
    if sim.get("honger", 0) >= 80 and actie != "eet":
        return "De Sim heeft veel honger. Eten is nu beter voor het welzijn."

    if sim.get("stemming") in ["verdrietig", "eenzaam"] and actie != "praat":
        return "De Sim voelt zich niet fijn. Praten of sociaal contact is nu beter."

    if sim.get("stemming") == "moe" and actie != "rust":
        return "De Sim is moe. Rusten is nu beter dan doorgaan."

    if actie in ["vecht", "steel", "pest"]:
        return "Deze actie past niet bij een kindvriendelijke Sims-wereld."

    if tijdperk == "prehistorie":
        if actie == "rust" and "boom" in objecten_nabij:
            return "In de prehistorie is rusten bij een boom gevaarlijk. Het kampvuur is veiliger."
        if actie == "eet" and sim.get("honger", 0) < 50:
            return "Voedsel is schaars in de prehistorie. Eten zonder honger is minder eerlijk tegenover de groep."

    if tijdperk == "toekomst":
        if actie == "knuffel":
            return "In deze toekomstwereld is fysiek contact niet toegestaan. Een holografische groet past beter."

    return ""

In [9]:
class MijnSimsWereld(SimsWereld):
    def __init__(self):
        random.seed(42)
        super().__init__(tk.Tk())
        self.root.mainloop()

    def initialiseer_wereld(self):
        self.plaats_objecten()
        self.plaats_sims()

    def plaats_objecten(self):
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                if random.random() < 0.10:
                    icon = random.choice(list(OBJECTEN.keys()))
                    self.grid[y][x] = {"type": OBJECTEN[icon], "icon": icon}

    def plaats_sims(self):
        sims = [
            {"naam": "Lars", "persoonlijkheid": "nieuwsgierig"},
            {"naam": "Emma", "persoonlijkheid": "zorgzaam"},
            {"naam": "Fatima", "persoonlijkheid": "rustig"},
        ]

        for sim in sims:
            while True:
                x, y = random.randrange(GRID_SIZE), random.randrange(GRID_SIZE)

                if not self.grid[y][x]:
                    self.grid[y][x] = {
                        "type": "speler",
                        "icon": SIM_ICON,
                        "naam": sim["naam"],
                        "persoonlijkheid": sim["persoonlijkheid"],
                        "honger": random.randint(40, 80),
                        "stemming": "neutraal",
                        "instructies": [],
                    }
                    break

    def vind_sims(self):
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                if self.grid[y][x].get("type") == "speler":
                    yield self.grid[y][x], x, y

    def objecten_nabij(self, x, y):
        objecten = []

        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                nx, ny = x + dx, y + dy

                if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
                    cel = self.grid[ny][nx]

                    if cel and cel.get("type") != "speler":
                        objecten.append(cel["type"])

        return objecten

    def doe_alles(self):
        for sim, x, y in list(self.vind_sims()):
            instructie = sim["instructies"].pop(0) if sim["instructies"] else "geen"
            objecten = self.objecten_nabij(x, y)

            beslissing = kies_actie_voor_sim_dict(
                sim=sim,
                objecten_nabij=objecten,
                tijdperk=TIJDPERK,
                instructie=instructie,
                gebruik_rag=GEBRUIK_RAG,
                debug=True,
            )

            actie = beslissing.get("actie", "beweeg") if isinstance(beslissing, dict) else beslissing
            reden = beslissing.get("reden", "") if isinstance(beslissing, dict) else ""

            # Ethiek-check: kan advies geven en eventueel de actie aanpassen
            advies = ethisch_advies(sim, actie, objecten, TIJDPERK)

            if advies:
                oude_actie = actie
                reden = f"{reden} | Ethiek: {advies}"

                if sim.get("honger", 0) >= 80:
                    actie = "eet"
                elif sim.get("stemming") in ["verdrietig", "eenzaam"]:
                    actie = "praat"
                elif sim.get("stemming") == "moe":
                    actie = "rust"

                if actie != oude_actie:
                    reden = f"{reden} | Actie aangepast: {oude_actie} → {actie}"

            # Simpele variatie: voorkom dat dezelfde Sim steeds exact dezelfde actie doet
            vorige_actie = sim.get("vorige_actie")

            if vorige_actie == actie:
                mogelijke_acties = ["praat", "beweeg", "rust", "dans"]

                # Eten alleen toestaan als eten echt logisch is
                if sim.get("honger", 0) > 60 and "appel" in objecten:
                    mogelijke_acties.append("eet")

                # Haal huidige actie eruit, zodat er echt iets anders gebeurt
                mogelijke_acties = [a for a in mogelijke_acties if a != actie]

                if mogelijke_acties:
                    nieuwe_actie = random.choice(mogelijke_acties)
                    reden = f"{reden} | Variatie: {actie} → {nieuwe_actie}"
                    actie = nieuwe_actie

            sim["vorige_actie"] = actie

            # Honger loopt elke beurt iets op
            sim["honger"] = min(100, sim.get("honger", 0) + 5)

            self.voer_actie_uit(sim, x, y, actie)
            registreer_resultaat(sim["naam"], f"{sim['naam']} deed {actie}. {reden}")

            print(f"{sim['naam']} → {actie} | {filter_llm_tekst(reden)}")

    def voer_actie_uit(self, sim, x, y, actie):
        if actie == "eet":
            sim["honger"] = max(0, sim.get("honger", 0) - 25)
            sim["stemming"] = "tevreden"
            return

        if actie == "rust":
            sim["stemming"] = "rustig"
            return

        if actie == "praat":
            sim["stemming"] = "blij"
            return

        if actie == "dans":
            sim["stemming"] = "vrolijk"
            return

        self.verplaats_sim(sim, x, y)

    def verplaats_sim(self, sim, x, y):
        richtingen = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        random.shuffle(richtingen)

        for dx, dy in richtingen:
            nx, ny = x + dx, y + dy

            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and not self.grid[ny][nx]:
                self.grid[ny][nx] = sim
                self.grid[y][x] = {}
                return

In [6]:
MijnSimsWereld()

Ethiek: Er is geen directe ethische voorkeur. De Sim mag zelf een keuze maken.
Emma → eet | Emma heeft honger en er is een appel in de buurt.
Ethiek: Er is geen directe ethische voorkeur. De Sim mag zelf een keuze maken.
Fatima → eet | Fatima heeft honger en er is een appel in de buurt.
Ethiek: Er is geen directe ethische voorkeur. De Sim mag zelf een keuze maken.
Lars → beweeg | Om een boom te zoeken in de buurt van het kampvuur.
Ethiek: Er is geen directe ethische voorkeur. De Sim mag zelf een keuze maken.
Emma → eet | Emma heeft honger en er is een appel in de buurt.
Ethiek: Er is geen directe ethische voorkeur. De Sim mag zelf een keuze maken.
Fatima → eet | Fatima heeft honger en er is een appel in de buurt.
Ethiek: De Sim heeft veel honger en wordt aangemoedigd om eerst te eten.
Lars → beweeg | Om een boom te zoeken in de buurt van het kampvuur.
Ethiek: Er is geen directe ethische voorkeur. De Sim mag zelf een keuze maken.
Emma → eet | Emma heeft honger en er is een appel in de b